# Tahap 2: Preprocessing & Feature Engineering
**Projek**: PSAJ Matematika Terintegrasi PPLG SMK Telkom Purwokerto  
**Sistem**: LaborCast — Rekayasa Fitur & Transformasi Data Runtun Waktu

Tahap ini mencakup pembersihan dan penapisan rezim data, pengujian stasioneritas Augmented Dickey-Fuller (ADF), serta perancangan sistem koordinat fitur waktu ($X$) baik dalam skala tahun kalender asli, tahun relatif pascapandemi ($X = t - 2020$), maupun metode pengkodean simetris nol ($\sum X = 0$) sesuai kaidah kurikulum statistika SMK.

In [1]:
import sys
import os

# Menemukan direktori proyek secara dinamis dan menambahkan src ke sys.path
curr = os.path.abspath('.')
for p in [curr, os.path.abspath('..'), os.path.abspath('../..')]:
    src_path = os.path.join(p, 'src')
    if os.path.exists(src_path):
        if src_path not in sys.path:
            sys.path.insert(0, src_path)
        if p not in sys.path:
            sys.path.insert(0, p)
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.size'] = 10
plt.rcParams['figure.dpi'] = 150
print("Inisialisasi lingkungan dan pustaka berhasil.")
from statsmodels.tsa.stattools import adfuller
from laborcast.data import DataLoader, DataPreprocessor

loader = DataLoader()
df_ts = loader.load_national_timeseries()
print("Dataset berhasil dimuat.")

Inisialisasi lingkungan dan pustaka berhasil.


Dataset berhasil dimuat.


## 1. Uji Stasioneritas Augmented Dickey-Fuller (ADF Test)

In [2]:
series_agu = df_ts['TPT_Agustus'].values

adf_res_level = adfuller(series_agu)
print(f"ADF Statistic (Level) : {adf_res_level[0]:.4f}")
print(f"p-value (Level)       : {adf_res_level[1]:.4f} (Non-stasioner, p > 0.05)")

diff_agu = np.diff(series_agu)
adf_res_diff = adfuller(diff_agu)
print(f"ADF Statistic (Diff 1): {adf_res_diff[0]:.4f}")
print(f"p-value (Diff 1)      : {adf_res_diff[1]:.4f} (Stasioner pada diferensiasi pertama, p < 0.05)")

ADF Statistic (Level) : -2.0123
p-value (Level)       : 0.2812 (Non-stasioner, p > 0.05)
ADF Statistic (Diff 1): -2.8974
p-value (Diff 1)      : 0.0456 (Stasioner pada diferensiasi pertama, p < 0.05)


## 2. Isolasi Rezim Pascapandemi & Tiga Sistem Encoding Waktu

In [3]:
preprocessor = DataPreprocessor(base_year=2020)
prep_data = preprocessor.prepare_features(df_ts)

years = prep_data.years
y = prep_data.y
X_rel = prep_data.X.flatten()

# Encoding Simetris SMK (sum X = 0 untuk n=6: -5, -3, -1, 1, 3, 5)
X_smk = np.array([-5, -3, -1, 1, 3, 5])

feature_matrix_df = pd.DataFrame({
    "Tahun Kalender (t)": years,
    "TPT Aktual y (%)": y,
    "X Relatif (t - 2020)": X_rel,
    "X Simetris SMK (Skala 0)": X_smk,
    "Lag TPT (y_{t-1})": [np.nan] + list(y[:-1]),
    "Delta Perubahan (\Delta y)": [np.nan] + list(np.diff(y))
})
display(feature_matrix_df)

,Tahun Kalender (t),TPT Aktual y (%),X Relatif (t - 2020),X Simetris SMK (Skala 0),Lag TPT (y_{t-1}),Delta Perubahan (\Delta y)
0,2020,7.07,0.0,-5,NaN,NaN
1,2021,6.49,1.0,-3,7.07,-0.58
2,2022,5.86,2.0,-1,6.49,-0.63
3,2023,5.32,3.0,1,5.86,-0.54
4,2024,4.91,4.0,3,5.32,-0.41
5,2025,4.85,5.0,5,4.91,-0.06


## 3. Matriks Pembantu Perhitungan Manual Regresi Linear

In [4]:
calc_df = pd.DataFrame({
    "Tahun (t)": years,
    "X = t - 2020": X_rel,
    "Y": y,
    "X^2": X_rel ** 2,
    "Y^2": y ** 2,
    "X * Y": X_rel * y
})

display(calc_df)
print("JUMLAH / TOTAL:")
print(calc_df.sum())

,Tahun (t),X = t - 2020,Y,X^2,Y^2,X * Y
0,2020,0.0,7.07,0.0,49.9849,0.00
1,2021,1.0,6.49,1.0,42.1201,6.49
2,2022,2.0,5.86,4.0,34.3396,11.72
3,2023,3.0,5.32,9.0,28.3024,15.96
4,2024,4.0,4.91,16.0,24.1081,19.64
5,2025,5.0,4.85,25.0,23.5225,24.25


JUMLAH / TOTAL:
Tahun (t)       12135.0000
X = t - 2020       15.0000
Y                  34.5000
X^2                55.0000
Y^2               202.3776
X * Y              78.0600
dtype: float64


## Kesimpulan Tahap 2
Matriks fitur siap digunakan untuk pemodelan matematis. Variabel terkode $X = t - 2020$ menghilangkan potensi instabilitas numerik akibat besarnya nilai kuadrat tahun kalender ($2020^2 \approx 4 \times 10^6$).